In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder


In [3]:
df = pd.read_csv("../../processed/combined.csv")
df= df[["statement", "status"]]
df = df.dropna()

df.head()

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety


In [4]:
df["status"].value_counts()

status
Normal                  16343
Depression              15404
Suicidal                10651
Anxiety                  3841
Bipolar                  2777
Stress                   2587
Personality disorder     1077
Name: count, dtype: int64

In [5]:
encoder = LabelEncoder()
y = encoder.fit_transform(df["status"]) 

In [9]:
vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["statement"]).toarray()

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(np.unique(y)), activation='softmax')
])  

c:\Users\mgv05\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [13]:
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/5
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.7164 - loss: 0.7802 - val_accuracy: 0.7863 - val_loss: 0.5644
Epoch 2/5
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8125 - loss: 0.4855 - val_accuracy: 0.7906 - val_loss: 0.5462
Epoch 3/5
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8503 - loss: 0.3863 - val_accuracy: 0.7896 - val_loss: 0.5557
Epoch 4/5
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8818 - loss: 0.3311 - val_accuracy: 0.7857 - val_loss: 0.6056
Epoch 5/5
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.9071 - loss: 0.2482 - val_accuracy: 0.7811 - val_loss: 0.6383


In [14]:
model.save("neural_network.model.h5")
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)